# Employee Activity Anomaly Detection - Training

In [ ]:
!pip install --upgrade pip
!pip install tensorflow scikit-learn pandas numpy matplotlib
!pip install onnx==1.17.0 onnxruntime==1.19.2 tf2onnx==1.16.1 protobuf==5.28.3
!pip install --upgrade pip
!pip install tensorflow==2.17.0 scikit-learn pandas numpy matplotlib
!pip install onnx==1.16.0 onnxruntime==1.18.0 tf2onnx==1.16.1

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import time
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score

from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation

import tensorflow as tf
import tf2onnx
import onnx
import onnxruntime as rt

In [ ]:
feature_indexes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
label_indexes = [11]

df_train = pd.read_csv('../data/train.csv')
X_train = df_train.iloc[:, feature_indexes].values
y_train = df_train.iloc[:, label_indexes].values

df_val = pd.read_csv('../data/validate.csv')
X_val = df_val.iloc[:, feature_indexes].values
y_val = df_val.iloc[:, label_indexes].values

df_test = pd.read_csv('../data/test.csv')
X_test = df_test.iloc[:, feature_indexes].values
y_test = df_test.iloc[:, label_indexes].values

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

Path("../artifact").mkdir(parents=True, exist_ok=True)
with open("../artifact/test_data.pkl", "wb") as f:
    pickle.dump((X_test, y_test), f)
with open("../artifact/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [ ]:
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train.ravel()
)
class_weights = {i: class_weights[i] for i in range(len(class_weights))}
print(f"Class weights: {class_weights}")

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=len(feature_indexes)))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
start_time = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

training_time = time.time() - start_time
print(f"\nTraining time: {training_time:.2f}s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None, len(feature_indexes)], tf.float32, name='dense_input')])
def model_fn(x):
    return model(x)

model_proto, _ = tf2onnx.convert.from_function(
    model_fn,
    input_signature=[tf.TensorSpec([None, len(feature_indexes)], tf.float32, name='dense_input')]
)

os.makedirs("../models/employee_anomaly/1", exist_ok=True)
onnx.save(model_proto, "../models/employee_anomaly/1/model.onnx")
print("Model saved")

In [ ]:
!ls -lh ../models/employee_anomaly/1/

In [ ]:
sess = rt.InferenceSession("../models/employee_anomaly/1/model.onnx", providers=rt.get_available_providers())
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

y_pred_proba = sess.run([output_name], {input_name: X_test.astype(np.float32)})[0]
y_pred = (np.squeeze(y_pred_proba) > 0.5).astype(int)
y_test_flat = y_test.ravel()

In [ ]:
acc = (y_pred == y_test_flat).sum() / len(y_test_flat)
precision = precision_score(y_test_flat, y_pred, zero_division=0)
recall = recall_score(y_test_flat, y_pred, zero_division=0)
f1 = f1_score(y_test_flat, y_pred, zero_division=0)

print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)

In [ ]:
cm = confusion_matrix(y_test_flat, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly']).plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

print(f"TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"FN: {cm[1,0]}, TP: {cm[1,1]}")

In [ ]:
def test_prediction(features, desc):
    scaled = scaler.transform([features]).astype(np.float32)
    prob = sess.run([output_name], {input_name: scaled})[0][0][0]
    status = "🚨 ANOMALY" if prob > 0.5 else "✅ Normal"
    print(f"\n{desc}")
    print(f"  Prob: {prob:.4f} {status}")

print("\n" + "="*50)
print("EXAMPLES")
print("="*50)

test_prediction([192, 168, 1, 45, 10, 0, 0, 1, 0, 0, 0], "Normal office login")
test_prediction([85, 232, 45, 100, 23, 1, 0, 0, 3, 1, 0], "Night download external IP")
test_prediction([10, 10, 10, 10, 14, 0, 0, 1, 5, 0, 1], "Failed logins")
test_prediction([192, 168, 1, 78, 14, 0, 0, 1, 2, 1, 0], "Normal document read")